# LAB 09 — K-Nearest Neighbors (KNN)
**Subject:** Artificial Intelligence     
**Name:** Abubakar Ahmad      
**SAP-ID:** 54603
---

### Learning Objectives
- Understand what KNN is
- Implement KNN from scratch
- Test KNN on different datasets with different K values
- Calculate and compare accuracy

## 1. What is KNN?

KNN (K-Nearest Neighbors) is a **non-parametric, lazy learning** algorithm.  
It does not build a model during training — it simply stores the training data and classifies new points based on proximity.

**Steps:**
1. Calculate Euclidean distance from the new point to all training points
2. Pick the K nearest neighbors
3. Assign the majority class label among those K neighbors

## 2. Importing Libraries

In [1]:
# Importing all required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import mode
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.metrics import accuracy_score

## 3. Building KNN Classifier from Scratch

We implement the same class structure shown in the lab.  
This helps us understand exactly how KNN works internally.

In [2]:
# K Nearest Neighbors Classification — Custom Implementation
class K_Nearest_Neighbors_Classifier():

    def __init__(self, K):
        # Store the value of K (number of neighbors to vote)
        self.K = K

    def fit(self, X_train, Y_train):
        # KNN is a lazy learner — no actual training happens
        # We just store the training data to use later during prediction
        self.X_train = X_train
        self.Y_train = Y_train
        # no_of_training_examples, no_of_features
        self.m, self.n = X_train.shape

    def predict(self, X_test):
        self.X_test  = X_test
        self.m_test, self.n = X_test.shape

        # Array to store predicted labels
        Y_predict = np.zeros(self.m_test)

        for i in range(self.m_test):
            x = self.X_test[i]
            # Find K nearest neighbors for each test point
            neighbors = self.find_neighbors(x)
            # Assign the most common class among neighbors (majority vote)
            Y_predict[i] = mode(neighbors)[0]

        return Y_predict

    def find_neighbors(self, x):
        # Calculate euclidean distance from x to every training point
        euclidean_distances = np.zeros(self.m)

        for i in range(self.m):
            d = self.euclidean(x, self.X_train[i])
            euclidean_distances[i] = d

        # Sort distances and get the indices of K nearest points
        inds = euclidean_distances.argsort()
        Y_train_sorted = self.Y_train[inds]

        # Return the labels of the K nearest neighbors
        return Y_train_sorted[:self.K]

    def euclidean(self, x, x_train):
        # Standard Euclidean distance: sqrt( sum( (x - x_train)^2 ) )
        return np.sqrt(np.sum(np.square(x - x_train)))

## 4. Dataset 1 — Iris Dataset

The Iris dataset contains **150 samples**, **4 features**, and **3 classes** (flower types).  
We start with K = 3 to match the lab example, then test multiple K values.

In [5]:
# Load Iris dataset
iris = load_iris()
# display(iris.data)
X    = iris.data    # Features: sepal/petal length and width
Y    = iris.target  # Labels: 0, 1, 2 (three flower classes)

# Splitting dataset into train and test set (67% train, 33% test)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=1/3, random_state=0
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

Training samples : 100
Testing  samples : 50
Features         : 4


In [4]:
# Model training with K = 3
model  = K_Nearest_Neighbors_Classifier(K=3)
model.fit(X_train, Y_train)

# Also training sklearn's KNN for comparison
model1 = KNeighborsClassifier(n_neighbors=3)
model1.fit(X_train, Y_train)

# Prediction on test set
Y_pred  = model.predict(X_test)
Y_pred1 = model1.predict(X_test)

# Measure performance
correctly_classified  = 0
correctly_classified1 = 0

for count in range(np.size(Y_pred)):
    if Y_test[count] == Y_pred[count]:
        correctly_classified  += 1
    if Y_test[count] == Y_pred1[count]:
        correctly_classified1 += 1

print("Accuracy on test set by our model    :", correctly_classified  / X_test.shape[0] * 100)
print("Accuracy on test set by sklearn model:", correctly_classified1 / X_test.shape[0] * 100)

Accuracy on test set by our model    : 96.0
Accuracy on test set by sklearn model: 96.0


## 5. Effect of Different K Values — Iris Dataset

Now we test K = 1, 3, 5, 7, 9 to see how the choice of K affects accuracy.

In [6]:
# Testing different values of K on Iris dataset
print(f"{'K':<8} {'Our Model':>14} {'Sklearn':>14}")
print("-" * 38)

for k in [1, 3, 5, 7, 9]:
    # Our custom model
    m = K_Nearest_Neighbors_Classifier(K=k)
    m.fit(X_train, Y_train)
    acc_ours = accuracy_score(Y_test, m.predict(X_test)) * 100

    # Sklearn model
    m2 = KNeighborsClassifier(n_neighbors=k)
    m2.fit(X_train, Y_train)
    acc_sk = accuracy_score(Y_test, m2.predict(X_test)) * 100

    print(f"K = {k:<4}  {acc_ours:>12.2f}%  {acc_sk:>12.2f}%")

K             Our Model        Sklearn
--------------------------------------
K = 1            96.00%         96.00%
K = 3            96.00%         96.00%
K = 5            98.00%         98.00%
K = 7            98.00%         98.00%
K = 9            98.00%         98.00%


## 6. Dataset 2 — Breast Cancer Dataset (Different Dataset)

The Breast Cancer dataset has **569 samples**, **30 features**, and **2 classes** (malignant / benign).  
We test the same K values to compare results across a different dataset.

In [7]:
# Load Breast Cancer dataset
data = load_breast_cancer()
X2   = data.data
Y2   = data.target

# Splitting dataset into train and test set
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    X2, Y2, test_size=1/3, random_state=0
)

print(f"Training samples : {X_train2.shape[0]}")
print(f"Testing  samples : {X_test2.shape[0]}")
print(f"Features         : {X_train2.shape[1]}")

Training samples : 379
Testing  samples : 190
Features         : 30


In [8]:
# Testing different values of K on Breast Cancer dataset
print(f"{'K':<8} {'Our Model':>14} {'Sklearn':>14}")
print("-" * 38)

for k in [1, 3, 5, 7, 9]:
    # Our custom model
    m = K_Nearest_Neighbors_Classifier(K=k)
    m.fit(X_train2, Y_train2)
    acc_ours = accuracy_score(Y_test2, m.predict(X_test2)) * 100

    # Sklearn model
    m2 = KNeighborsClassifier(n_neighbors=k)
    m2.fit(X_train2, Y_train2)
    acc_sk = accuracy_score(Y_test2, m2.predict(X_test2)) * 100

    print(f"K = {k:<4}  {acc_ours:>12.2f}%  {acc_sk:>12.2f}%")

K             Our Model        Sklearn
--------------------------------------
K = 1            92.11%         92.11%
K = 3            93.16%         93.16%
K = 5            94.74%         94.74%
K = 7            95.79%         95.79%
K = 9            96.32%         96.32%


## 7. Results Comparison & Conclusion

| Dataset        | Best K | Best Accuracy |
|----------------|--------|---------------|
| Iris           | K = 5  | 98.00%        |
| Breast Cancer  | K = 9  | 96.32%        |

**Observations:**
- Our custom KNN matched Sklearn exactly on every K value — confirming the implementation is correct.
- On the **Iris dataset**, accuracy improved from 96% at K=1 to 98% at K=5 and stayed stable.
- On the **Breast Cancer dataset**, accuracy increased steadily with K, reaching 96.32% at K=9.
- **Smaller K** → sensitive to noise (can overfit).
- **Larger K** → smoother boundaries but may underfit if too large.
- The right K depends on the dataset and should be chosen by testing multiple values.